In [ ]:
def gameLoop(data, sock):
    analog_stick = Arduino_Analog(base.ARDUINO, [0, 1])

    y = 10
    x = 10
    prox = "null"
    win = False
    lose = False
    blink = False

    max_x = 127
    max_y = 127
    
    while win == False and lose == False:
        rec = sock.recv(1024)
        
        if (rec.decode() == '1'):
            lose = True

        else:
            vals = analog_stick.read() 

            x_val = vals[0]
            y_val = vals[1]

            y_val_map = map_val(y_val, 0, 3.3, 1, -1)
            x_val_map = map_val(x_val, 0, 3.3, -1, 1)

            y_val_map = round(y_val_map, 2)
            x_val_map = round(x_val_map, 2)

            direc = direction(x_val_map, y_val_map)

            coordinatex = 100
            coordinatey = 100

            if direc == 1:
                x = x + 1
                time.sleep(.1)
            if direc == 2:
                x = x - 1
                time.sleep(.1)
            if direc == 3:
                y = y - 1
                time.sleep(.1)
            if direc == 4:
                y = y + 1
                time.sleep(.1)

            x = max(0,min(x,max_x))
            y = max(0,min(y,max_y))

            if data[y,x,0] == 255:
                prox = "BLANK "
                write_gpio(3,0)
                write_gpio(2,0)

            if data[y,x,0] < 255 and data[y,x,0] > 127:
                prox = "GREEN "
                write_gpio(3,1)
                write_gpio(2,0)

            if data[y,x,0] < 128 and data[y,x,0] > 63:
                prox = "YELLOW"
                write_gpio(3,1)
                write_gpio(2,1)

            if data[y,x,0] < 64 and data[y,x,0] > 31:
                prox = "RED   "
                write_gpio(3,0)
                write_gpio(2,1)

            if data[y,x,0] < 32:
                prox = "DIG   "
                blink = not blink
                if blink:
                    write_gpio(2,1)
                if not blink:
                    write_gpio(2,0)
                if (btns[0].read() == 1):
                    win = True
                    msg = '1'
                    b_en = msg.encode(encoding= "utf-8")
                    sock.sendall(b_en)

            print(f"X: {x:2d} Y: {y:2d} Proximity: {prox}", end="\r")
            time.sleep(.1)
            ##except: # receive winning msg from other player
            ##   print("You lost! Game over...")
            
    if win == True:
        print("You found the treasure!!")
        write_gpio(2,0)
        for i in range(5):
            write_gpio(1,1)
            time.sleep(0.75)
            write_gpio(1,0)
            time.sleep(0.75)
    else:
        print("You lost! Game over...")
        write_gpio(2,0)
        for i in range(5):
            write_gpio(2,1)
            time.sleep(1)
            write_gpio(2,0)
            time.sleep(1)